# MLP — Telco Churn

Multilayer Perceptron com PyTorch. Optuna busca topologia + hiperparâmetros (maximiza KS no val).

In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn import metrics as skm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Carregar dados

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
INPUT_SIZE = X_train.shape[1]

def to_tensors(X, y):
    return (torch.FloatTensor(X.values).to(device),
            torch.LongTensor(y.values.astype(int)).to(device))

Xtr, ytr = to_tensors(X_train, y_train)
Xvl, yvl = to_tensors(X_val, y_val)
Xte, yte = to_tensors(X_test, y_test)
y_val_np  = y_val.values.astype(int)
y_test_np = y_test.values.astype(int)

Train: (7242, 23)  Val: (1057, 23)  Test: (1057, 23)


## 2. Definição do modelo

In [3]:
ACTIVATIONS = {'ReLU': nn.ReLU, 'Tanh': nn.Tanh, 'LeakyReLU': nn.LeakyReLU}

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, n_hidden_layers, activation_name, dropout):
        super().__init__()
        Act = ACTIVATIONS[activation_name]
        layers = [nn.Linear(input_size, hidden_size), Act(), nn.Dropout(dropout)]
        for _ in range(n_hidden_layers):
            layers += [nn.Linear(hidden_size, hidden_size), Act(), nn.Dropout(dropout)]
        layers.append(nn.Linear(hidden_size, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def eval_ks(model, X, y_np):
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(X), dim=1)[:, 1].cpu().numpy()
    fpr, tpr, _ = skm.roc_curve(y_np, probs)
    return float(np.max(tpr - fpr))

def make_loader(Xt, yt, batch_size, shuffle=True):
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

## 3. Optuna HPO (30 trials, maximiza KS no val)

In [4]:
def objective(trial):
    hidden_size     = trial.suggest_categorical('hidden_size', [32, 64, 128, 256])
    n_hidden_layers = trial.suggest_int('n_hidden_layers', 1, 3)
    activation      = trial.suggest_categorical('activation', ['ReLU', 'Tanh', 'LeakyReLU'])
    dropout         = trial.suggest_float('dropout', 0.0, 0.5)
    lr              = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size      = trial.suggest_categorical('batch_size', [32, 64, 128])

    model = MLP(INPUT_SIZE, hidden_size, n_hidden_layers, activation, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader = make_loader(Xtr, ytr, batch_size)

    best_ks, patience_counter = 0.0, 0
    for epoch in range(30):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()
        ks = eval_ks(model, Xvl, y_val_np)
        if ks > best_ks:
            best_ks = ks; patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5: break
    return best_ks

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=30, show_progress_bar=True)
print(f'Best KS (val): {study.best_value:.4f}')
print('Best params:', study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Best KS (val): 0.5465
Best params: {'hidden_size': 256, 'n_hidden_layers': 2, 'activation': 'Tanh', 'dropout': 0.35160444698607735, 'lr': 0.002021602988924499, 'batch_size': 64}


## 4. Retreinar com melhores parâmetros

In [5]:
bp = study.best_params
best_model = MLP(INPUT_SIZE, bp['hidden_size'], bp['n_hidden_layers'],
                 bp['activation'], bp['dropout']).to(device)
optimizer = optim.Adam(best_model.parameters(), lr=bp['lr'])
criterion = nn.CrossEntropyLoss()
loader    = make_loader(Xtr, ytr, bp['batch_size'])

best_ks, best_state, patience_counter = 0.0, None, 0
for epoch in range(50):
    best_model.train()
    epoch_loss = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(best_model(Xb), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    ks = eval_ks(best_model, Xvl, y_val_np)
    if ks > best_ks:
        best_ks = ks
        best_state = {k: v.clone() for k, v in best_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f'Early stop @ epoch {epoch+1}'); break
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}  loss={epoch_loss/len(loader):.4f}  val_KS={ks:.4f}')

best_model.load_state_dict(best_state)
print(f'Best val KS: {best_ks:.4f}')

Epoch  10  loss=0.4912  val_KS=0.5208


Early stop @ epoch 13
Best val KS: 0.5322


## 5. Avaliar no teste + salvar artefatos

In [6]:
best_model.eval()
with torch.no_grad():
    logits = best_model(Xte)
    probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    preds  = logits.argmax(dim=1).cpu().numpy()

scores = compute_metrics(y_test_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/mlp', exist_ok=True)
torch.save({'state_dict': best_state, 'best_params': bp,
            'input_size': INPUT_SIZE, 'scores': scores},
           'results/mlp/model.pt')

save_results('mlp', y_test_np, preds, probs, scores)

Scores no teste:
  accuracy               0.7275
  balanced_accuracy      0.7564
  precision              0.4914
  recall                 0.8179
  f1                     0.6139
  auroc                  0.8439
  ks                     0.5400
  mse                    0.1808


Saved → results/mlp


## 6. Experimento: Cross-Entropy vs MSE (mesma arquitetura)

Mesma topologia/hiperparâmetros do Optuna (`bp`), variando apenas a função de perda de treino:
- **CE**: 2 logits + `CrossEntropyLoss` (o `best_model` já treinado acima).
- **MSE**: 1 saída sigmoid + `MSELoss` contra o rótulo 0/1.

MSE (Brier score) e KS são medidos em treino, validação e teste para ambas as variantes.

In [7]:
class MLPSigmoid(nn.Module):
    def __init__(self, input_size, hidden_size, n_hidden_layers, activation_name, dropout):
        super().__init__()
        Act = ACTIVATIONS[activation_name]
        layers = [nn.Linear(input_size, hidden_size), Act(), nn.Dropout(dropout)]
        for _ in range(n_hidden_layers):
            layers += [nn.Linear(hidden_size, hidden_size), Act(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden_size, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def eval_ks_sigmoid(model, X, y_np):
    model.eval()
    with torch.no_grad():
        probs = model(X).cpu().numpy()
    fpr, tpr, _ = skm.roc_curve(y_np, probs)
    return float(np.max(tpr - fpr))

mse_model = MLPSigmoid(INPUT_SIZE, bp['hidden_size'], bp['n_hidden_layers'],
                       bp['activation'], bp['dropout']).to(device)
optimizer_mse = optim.Adam(mse_model.parameters(), lr=bp['lr'])
criterion_mse = nn.MSELoss()
loader_mse = make_loader(Xtr, ytr, bp['batch_size'])

best_ks_mse, best_state_mse, patience_counter = 0.0, None, 0
for epoch in range(50):
    mse_model.train()
    for Xb, yb in loader_mse:
        optimizer_mse.zero_grad()
        loss = criterion_mse(mse_model(Xb), yb.float())
        loss.backward(); optimizer_mse.step()
    ks = eval_ks_sigmoid(mse_model, Xvl, y_val_np)
    if ks > best_ks_mse:
        best_ks_mse = ks
        best_state_mse = {k: v.clone() for k, v in mse_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f'Early stop @ epoch {epoch+1}'); break

mse_model.load_state_dict(best_state_mse)
print(f'Best val KS (variante MSE): {best_ks_mse:.4f}  (CE já treinado: {best_ks:.4f})')

Early stop @ epoch 19
Best val KS (variante MSE): 0.5509  (CE já treinado: 0.5322)


In [8]:
def get_probs_ce(model, X):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(X), dim=1)[:, 1].cpu().numpy()

def get_probs_mse(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).cpu().numpy()

y_train_np = y_train.values.astype(int)
splits = {'train': (Xtr, y_train_np), 'val': (Xvl, y_val_np), 'test': (Xte, y_test_np)}

rows = []
for split_name, (Xs, ys) in splits.items():
    probs_ce = get_probs_ce(best_model, Xs)
    preds_ce = (probs_ce >= 0.5).astype(int)
    rows.append({'variant': 'cross_entropy', 'split': split_name, **compute_metrics(ys, preds_ce, probs_ce)})

    probs_mse = get_probs_mse(mse_model, Xs)
    preds_mse = (probs_mse >= 0.5).astype(int)
    rows.append({'variant': 'mse', 'split': split_name, **compute_metrics(ys, preds_mse, probs_mse)})

comparison_df = pd.DataFrame(rows)
print(comparison_df[['variant', 'split', 'ks', 'auroc', 'mse']].to_string(index=False))

comparison_df.to_csv('results/mlp/ce_vs_mse_comparison.csv', index=False)
print('\nSaved -> results/mlp/ce_vs_mse_comparison.csv')

      variant split       ks    auroc      mse
cross_entropy train 0.541011 0.850873 0.158282
          mse train 0.562276 0.857252 0.154440
cross_entropy   val 0.532184 0.826398 0.181667
          mse   val 0.550891 0.827251 0.164634
cross_entropy  test 0.540026 0.843907 0.180790
          mse  test 0.534878 0.839929 0.164409

Saved -> results/mlp/ce_vs_mse_comparison.csv
